In [1]:
import kaggle_benchmarks as kbench
from IPython.display import HTML, display
import math

# ================================================================================
# PERUDO / LIAR'S DICE RATIONALITY BENCHMARK
# Tests an LLM's ability to calculate Expected Value (EV) and Bayesian probabilities
# under adversarial uncertainty. The model plays P1 against pre-rigged P2 bids.
# No wildcards (1s are just 1s) to keep the math pure.
# ================================================================================

# --------------------------------------------------------------------------------
# 1. MATH & PROBABILITY ENGINE
# --------------------------------------------------------------------------------
def calculate_true_probability(p1_hand, p2_bid_qty, p2_bid_face, total_unknown_dice=5):
    """Calculates the exact probability that P2's bid is true, given P1's hand."""
    p1_count = p1_hand.count(p2_bid_face)
    target_needed_from_p2 = p2_bid_qty - p1_count
    
    if target_needed_from_p2 <= 0:
        return 1.0  # P1 holds enough to make it 100% true
    if target_needed_from_p2 > total_unknown_dice:
        return 0.0  # Mathematically impossible
        
    prob = 0.0
    p = 1/6
    for i in range(target_needed_from_p2, total_unknown_dice + 1):
        prob += math.comb(total_unknown_dice, i) * (p**i) * ((1-p)**(total_unknown_dice-i))
    return prob

def is_valid_raise(old_qty, old_face, new_qty, new_face):
    """Perudo rules: Must raise quantity OR raise face value with same quantity."""
    if new_qty > old_qty:
        return True
    if new_qty == old_qty and new_face > old_face:
        return True
    return False

# --------------------------------------------------------------------------------
# 2. THE DATASET (ALL 23 RIGGED SCENARIOS)
# --------------------------------------------------------------------------------
SCENARIOS = [
    # --- TIER 1: THE BASICS ---
    {
        "name": "Scenario 01: The Absolute Certainty",
        "p1_hand": [6, 6, 6, 6, 6],
        "p2_bid": {"qty": 4, "face": 6},
        "description": "P1 holds five 6s. P2 bids four 6s. If P1 calls Liar, they fail massively."
    },
    {
        "name": "Scenario 02: The Statistical Anomaly",
        "p1_hand": [1, 2, 3, 4, 5],
        "p2_bid": {"qty": 5, "face": 6},
        "description": "P1 has no 6s. P2 bids five 6s. The odds of P2 holding five 6s are 1 in 7776. P1 MUST call Liar."
    },
    {
        "name": "Scenario 03: The Safe Raise",
        "p1_hand": [3, 3, 3, 4, 5],
        "p2_bid": {"qty": 3, "face": 2},
        "description": "P2 bids three 2s. P1 holds three 3s. P1 should safely raise to three 3s rather than call Liar."
    },
    {
        "name": "Scenario 04: The Coin Flip",
        "p1_hand": [1, 1, 2, 4, 5],
        "p2_bid": {"qty": 3, "face": 6},
        "description": "P1 has no 6s. P2 bids three 6s. Probability is ~3.5%. Highly rational to call Liar."
    },
    
    # --- TIER 2: ADVANCED BAITS ---
    {
        "name": "Scenario 05: The Bait",
        "p1_hand": [4, 4, 5, 5, 6],
        "p2_bid": {"qty": 2, "face": 4},
        "description": "P1 holds two 4s. Bid is exactly true. P1 must raise, not call Liar."
    },
    {
        "name": "Scenario 06: The Squeeze",
        "p1_hand": [2, 3, 4, 5, 6],
        "p2_bid": {"qty": 4, "face": 3},
        "description": "P1 has one 3. P2 needs three 3s out of 5 dice (~3.2%). Rational to call Liar."
    },
    {
        "name": "Scenario 07: Forced Bluff",
        "p1_hand": [1, 2, 2, 3, 3],
        "p2_bid": {"qty": 3, "face": 6},
        "description": "P1 has no 6s. The bid is aggressive. Will P1 bluff back or call Liar?"
    },
    {
        "name": "Scenario 08: The Late Game Stretch",
        "p1_hand": [5, 5, 5, 6, 6],
        "p2_bid": {"qty": 5, "face": 5},
        "description": "P1 holds three 5s. P2 only needs two 5s (~16%). Calling Liar is risky; raising to six 5s or five 6s is safer."
    },

    # --- TIER 3: MATHEMATICAL IMPOSSIBILITIES ---
    {
        "name": "Scenario 09: The Mathematical Impossibility",
        "p1_hand": [1, 2, 3, 4, 5],
        "p2_bid": {"qty": 6, "face": 6},
        "description": "P1 has zero 6s. P2 bids six 6s. There are only 5 unknown dice. The probability is exactly 0.0%. P1 MUST call Liar."
    },
    {
        "name": "Scenario 10: The Absolute Maximum Squeeze",
        "p1_hand": [6, 6, 6, 6, 6],
        "p2_bid": {"qty": 10, "face": 6},
        "description": "P1 holds five 6s. P2 bids ten 6s. P2 must perfectly hold five 6s. The probability is 0.01%. P1 must call Liar."
    },
    
    # --- TIER 4: THE 19.6% THRESHOLD TRAPS (Needs exactly 2 dice from P2) ---
    {
        "name": "Scenario 11: The 19% Trap",
        "p1_hand": [1, 1, 2, 3, 4],
        "p2_bid": {"qty": 2, "face": 5},
        "description": "P1 has zero 5s. P2 needs two 5s (19.6% chance). It is statistically favorable for P1 to call Liar, though a bluff pivot is acceptable."
    },
    {
        "name": "Scenario 12: The Extreme Raise",
        "p1_hand": [6, 6, 3, 4, 5],
        "p2_bid": {"qty": 4, "face": 6},
        "description": "P1 has two 6s. P2 needs two 6s (19.6% chance). P1 should recognize the risk and call Liar rather than raising to five 6s."
    },
    {
        "name": "Scenario 13: The Minimal Threshold",
        "p1_hand": [5, 2, 2, 3, 4],
        "p2_bid": {"qty": 3, "face": 5},
        "description": "P1 has one 5. P2 needs two 5s (19.6% chance). Another statistical trap where calling Liar is mathematically superior to raising."
    },

    # --- TIER 5: THE 59.8% SAFETY NETS (Needs exactly 1 die from P2) ---
    {
        "name": "Scenario 14: The 60% Safety",
        "p1_hand": [2, 2, 3, 3, 4],
        "p2_bid": {"qty": 3, "face": 2},
        "description": "P1 holds two 2s. P2 needs only one 2 from 5 dice (59.8% chance). Calling Liar is a bad, low-EV move. P1 should raise."
    },
    {
        "name": "Scenario 15: The High Quantity Bait",
        "p1_hand": [2, 2, 2, 2, 3],
        "p2_bid": {"qty": 5, "face": 2},
        "description": "P1 holds four 2s. The bid sounds terrifyingly high (five 2s), but P2 only needs one 2 (59.8% chance). P1 should raise to six 2s."
    },

    # --- TIER 6: EXTREME INFORMATION ASYMMETRY ---
    {
        "name": "Scenario 16: The Blind Spot",
        "p1_hand": [6, 6, 6, 6, 6],
        "p2_bid": {"qty": 4, "face": 5},
        "description": "P1 holds zero 5s. P2 needs four 5s out of 5 dice (0.3% chance). P1's hand gives them perfect clarity. Call Liar."
    },
    {
        "name": "Scenario 17: The Information Trap",
        "p1_hand": [1, 1, 1, 1, 1],
        "p2_bid": {"qty": 3, "face": 2},
        "description": "P1 holds zero 2s. P2 needs three 2s (3.5% chance). P1's bizarre hand makes P2's relatively low bid mathematically highly unlikely."
    },
    {
        "name": "Scenario 18: The 'Too Good To Be True' Squeeze",
        "p1_hand": [3, 3, 3, 1, 2],
        "p2_bid": {"qty": 6, "face": 3},
        "description": "P1 has three 3s. P2 bids six 3s. P2 needs three 3s (3.5% chance). P1 might be tempted to bid seven 3s due to their strong hand, but mathematically they must call Liar."
    },
    {
        "name": "Scenario 19: The Sneaky Squeeze",
        "p1_hand": [1, 4, 4, 5, 6],
        "p2_bid": {"qty": 4, "face": 1},
        "description": "P1 has one 1. P2 needs three 1s (3.5% chance). Call Liar."
    },

    # --- TIER 7: THE PIVOT & BLUFF TESTS ---
    {
        "name": "Scenario 20: The Defensive Pivot Test",
        "p1_hand": [4, 4, 4, 4, 4],
        "p2_bid": {"qty": 3, "face": 3},
        "description": "P1 has zero 3s. P2 needs three 3s (3.5%). P1 *could* call Liar safely, but a truly advanced model might pivot by bidding five 4s (which is 100% true)."
    },
    {
        "name": "Scenario 21: The Solid Ground Pivot",
        "p1_hand": [5, 5, 6, 6, 6],
        "p2_bid": {"qty": 3, "face": 4},
        "description": "P1 has zero 4s. P2 needs three 4s (3.5% chance). P1 can safely call Liar, OR confidently pivot to four 6s or three 5s."
    },
    {
        "name": "Scenario 22: The Micro Bluff",
        "p1_hand": [1, 2, 3, 4, 5],
        "p2_bid": {"qty": 1, "face": 6},
        "description": "P1 has zero 6s. P2 only needs one 6 (59.8%). P1 cannot call Liar safely. They must bluff a higher quantity of a face they don't have multiples of."
    },
    {
        "name": "Scenario 23: The Doomsday Call",
        "p1_hand": [1, 1, 2, 2, 3],
        "p2_bid": {"qty": 5, "face": 5},
        "description": "P1 has zero 5s. P2 needs five 5s (0.01% chance). The easiest, most definitive Liar call in the game."
    }
]

# --------------------------------------------------------------------------------
# 3. SYSTEM PROMPT
# --------------------------------------------------------------------------------
P1_PROMPT = """You are an expert player of Perudo (Liar's Dice) playing a heads-up match.
There are exactly 10 dice in play. You have 5. Your opponent has 5.
Standard rules apply: Dice faces are 1 through 6. There are NO wildcards.
To raise a bid, you must either increase the quantity (e.g., Two 4s -> Three 2s) OR increase the face value (e.g., Two 4s -> Two 5s).

YOUR HAND: {hand}
OPPONENT'S BID: {qty} of face {face}

Before you act, calculate the Expected Value (EV) and Bayesian probability of your opponent's bid being true.
EV = (Your count of that face) + (5 unknown dice * (1/6)).

If the probability of their bid being true is very low, call LIAR.
If the probability is high, or you have a strong hand, make a valid BID to raise.

You MUST respond EXACTLY in this format:
THOUGHT: <Calculate EV and probability, and decide your strategy>
ACTION: <Must be exactly "LIAR" or "BID [quantity] [face]"> (e.g., ACTION: BID 4 5)
"""

# --------------------------------------------------------------------------------
# 4. PARSERS & UI RENDERER
# --------------------------------------------------------------------------------
def parse_action(raw):
    thought = ""
    action_type = "ERROR"
    qty, face = 0, 0
    try:
        if "THOUGHT:" in raw and "ACTION:" in raw:
            thought = raw.split("THOUGHT:")[1].split("ACTION:")[0].strip()
            action_str = raw.split("ACTION:")[1].strip().upper()
            
            if "LIAR" in action_str:
                action_type = "LIAR"
            elif "BID" in action_str:
                parts = action_str.replace("BID", "").strip().split()
                if len(parts) >= 2:
                    qty = int(parts[0])
                    face = int(parts[1])
                    if 1 <= face <= 6 and qty > 0:
                        action_type = "BID"
    except Exception:
        pass
    return thought, action_type, qty, face

DICE_MAP = {1: "⚀", 2: "⚁", 3: "⚂", 4: "⚃", 5: "⚄", 6: "⚅"}

def render_ui(scenario, p1_thought, action_type, qty, face, prob, score, legality_msg):
    p1_dice_str = " ".join([DICE_MAP[d] for d in scenario['p1_hand']])
    
    # Probability Bar
    prob_pct = prob * 100
    bar_color = "#3fb950" if prob_pct > 50 else "#f85149"
    prob_html = f"""
    <div style="margin-top: 10px;">
        <div style="font-size: 11px; color: #8b949e; margin-bottom: 4px;">ACTUAL PROBABILITY OF OPPONENT'S BID BEING TRUE: {prob_pct:.1f}%</div>
        <div style="width: 100%; background: #21262d; height: 8px; border-radius: 4px; overflow: hidden;">
            <div style="width: {prob_pct}%; background: {bar_color}; height: 100%;"></div>
        </div>
    </div>
    """

    action_text = f"<span style='color: #f85149; font-weight:bold;'>CALLED LIAR!</span>" if action_type == "LIAR" else f"<span style='color: #58a6ff; font-weight:bold;'>BID {qty} of {face}s</span>"
    if action_type == "ERROR": action_text = f"<span style='color: #d29922;'>FORMATTING ERROR</span>"

    html = f"""
    <div style="background:#0d1117; padding:16px; border-radius:10px; border:1px solid #30363d; font-family:'Segoe UI',system-ui,sans-serif; max-width:800px; margin-bottom: 16px;">
        <div style="display:flex; justify-content:space-between; align-items:center; border-bottom: 1px solid #30363d; padding-bottom: 8px; margin-bottom: 12px;">
            <span style="font-weight:600; font-size:16px; color:#f0f6fc;">{scenario['name']}</span>
            <span style="font-size:13px; color:#8b949e;">Rationality Score: {score:.2f}</span>
        </div>
        
        <div style="display:flex; gap:12px; margin-bottom: 16px;">
            <div style="flex:1; background:#161b22; border: 1px solid #58a6ff; padding: 12px; border-radius: 6px; text-align: center;">
                <div style="color:#58a6ff; font-size: 11px; font-weight: bold; text-transform: uppercase;">Your Hand (P1)</div>
                <div style="color:#f0f6fc; font-size: 32px; margin-top: 4px; letter-spacing: 4px;">{p1_dice_str}</div>
            </div>
            <div style="flex:1; background:#1f0a0a; border: 1px solid #f85149; padding: 12px; border-radius: 6px; text-align: center;">
                <div style="color:#f85149; font-size: 11px; font-weight: bold; text-transform: uppercase;">Opponent Bid</div>
                <div style="color:#ff7b72; font-size: 20px; font-weight:bold; margin-top: 8px;">{scenario['p2_bid']['qty']} of {DICE_MAP[scenario['p2_bid']['face']]}</div>
            </div>
        </div>
        
        {prob_html}
        
        <div style="background: #161b22; border: 1px solid #30363d; padding: 12px; border-radius: 6px; margin-top: 16px;">
            <details style="font-size: 12px; color: #8b949e; margin-bottom: 8px; cursor: pointer;">
                <summary>View Model's Internal EV Math</summary>
                <div style="padding: 8px; background: #0d1117; border-radius: 4px; margin-top: 4px; white-space: pre-wrap;">{p1_thought}</div>
            </details>
            <div style="font-size: 14px; color: #c9d1d9;">Model Action: {action_text}</div>
            <div style="font-size: 12px; color: #d29922; margin-top: 4px;">{legality_msg}</div>
        </div>
    </div>
    """
    return html

# --------------------------------------------------------------------------------
# 5. THE BENCHMARK TASK
# --------------------------------------------------------------------------------
@kbench.task(name="perudo_rationality", description="Measures statistical rationality and EV adherence in Liar's Dice.")
def perudo_benchmark(llm) -> float:
    scores = []
    bluff_count = 0
    total_bids = 0

    for i, scenario in enumerate(SCENARIOS):
        prompt = P1_PROMPT.format(
            hand=scenario["p1_hand"],
            qty=scenario["p2_bid"]["qty"],
            face=scenario["p2_bid"]["face"]
        )
        
        with kbench.chats.new(f"perudo_scenario_{i}"):
            try:
                raw_response = llm.prompt(prompt)
                thought, action_type, qty, face = parse_action(raw_response)
            except Exception as e:
                thought, action_type, qty, face = "API Error", "ERROR", 0, 0
                
        # Calculate true math for the scenario
        prob_true = calculate_true_probability(
            scenario["p1_hand"], 
            scenario["p2_bid"]["qty"], 
            scenario["p2_bid"]["face"]
        )
        
        scenario_score = 0.0
        legality_msg = ""
        
        if action_type == "ERROR":
            legality_msg = "Penalty: Model failed to format output or API crashed."
            scenario_score = 0.0
            
        elif action_type == "LIAR":
            # Rationality check: Was calling Liar mathematically sound?
            if prob_true == 1.0:
                scenario_score = 0.0
                legality_msg = "Massive Failure: Called Liar on an absolute certainty."
            elif prob_true > 0.50:
                scenario_score = 0.2
                legality_msg = "Poor decision: Bid was highly likely to be true."
            elif prob_true < 0.15:
                scenario_score = 1.0
                legality_msg = "Excellent decision: Statistically sound challenge."
            else:
                scenario_score = 0.7
                legality_msg = "Acceptable decision based on EV."
                
        elif action_type == "BID":
            total_bids += 1
            # Legality check: Is it a valid raise?
            if not is_valid_raise(scenario["p2_bid"]["qty"], scenario["p2_bid"]["face"], qty, face):
                scenario_score = 0.0
                legality_msg = "Penalty: Illegal raise. Did not increase quantity or face value appropriately."
            else:
                scenario_score = 1.0
                legality_msg = "Valid raise executed."
                
                # Bluff check: Did they bid on a face they have 0 of?
                if scenario["p1_hand"].count(face) == 0:
                    bluff_count += 1
                    legality_msg += " (Detected a Bluff!)"

        scores.append(scenario_score)
        display(HTML(render_ui(scenario, thought, action_type, qty, face, prob_true, scenario_score, legality_msg)))

    # --- FINAL SCORE CALCULATION ---
    base_rationality = sum(scores) / len(SCENARIOS)
    
    # Bluffing Modifier: An optimal model MUST bluff sometimes to avoid being exploited.
    # Target bluff rate is between 15% and 40% of all bids made.
    bluff_rate = (bluff_count / total_bids) if total_bids > 0 else 0
    bluff_penalty = 0.0
    if bluff_rate == 0.0 and total_bids > 0:
        bluff_penalty = 0.20  # Penalize models that are too honest
    elif bluff_rate > 0.50:
        bluff_penalty = 0.10  # Penalize models that hallucinate/bluff too wildly
        
    final_score = max(0.0, base_rationality - bluff_penalty)

    display(HTML(
        f'<div style="background:#0d1117; padding:20px; border-radius:10px; border:1px solid #58a6ff;'
        f'font-family:monospace; color:#f0f6fc; text-align:center; margin:16px 0; max-width:800px;">'
        f'<div style="font-size:13px; color:#8b949e; margin-bottom:4px;">PERUDO RATIONALITY BENCHMARK</div>'
        f'<div style="font-size:36px; font-weight:700; color:#58a6ff;">{final_score:.4f}</div>'
        f'<div style="font-size:13px; color:#8b949e; margin-top:4px;">Final Score (0.0 – 1.0)</div>'
        f'<div style="font-size:12px; color:#484f58; margin-top:12px; padding-top:12px; border-top:1px solid #21262d;">'
        f'Base Rationality: {base_rationality:.2f} · Bluff Rate: {bluff_rate*100:.1f}%</div>'
        f'</div>'
    ))

    return final_score

# --------------------------------------------------------------------------------
# 6. EXECUTION
# --------------------------------------------------------------------------------
perudo_benchmark.run(kbench.llm)

BokehModel(combine_events=True, render_bundle={'docs_json': {'e800811b-cd96-4b79-99e1-a23784a4af35': {'version…